# To perform and visualize analyses online 

## Before the experiment: get everything ready 

### imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Load config
from IPython.display import display
import sys
import os
from pathlib import Path


# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

from model_in_the_loop.utils.hydra_utils import load_config,set_env_vars
cfg = load_config()
set_env_vars(cfg)  # set env variables for repo and data paths
print(f"is_ensemble_model: {cfg.model_configs.is_ensemble_model}, only_train_readout: {cfg.model_configs.only_train_readout}")

home directory: /gpfs01/euler/User/ssuhai
is_ensemble_model: True, only_train_readout: True


In [3]:
from model_in_the_loop.core.dj_wrappers import (DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper)

from model_in_the_loop.utils.file_management import copy_rec_files,create_directory_structure,rm_all_experiment_dirs,clear_data_dump_dir
from model_in_the_loop.utils.transform_to_avi_stimulus import create_single_mei_avis_and_metadata,create_rf_test_dir
from model_in_the_loop.utils.simple_logging import log
from model_in_the_loop.utils.plotting import show_all_rois_plot

/workspace/.venv/lib/python3.12/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Create processing components (connect them to DB)

In [4]:
# create preprocessor
dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore

                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore
                )



In [6]:
dj_table_holder.setup()


[2025-10-26 21:07:39,043][INFO]: DataJoint is configured from /gpfs01/euler/User/ssuhai/datajoint/dj_ssuhai_conf.json


schema_name: ageuler_ssuhai_closed_loop
Done reconnecting. Skipping adding new entries from config.


/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/core/dj_wrappers/base.py:291: UserWarning: 
Some DJ tables (like UserInfo) are not empty, skipping adding new entries from config.
Make sure this is wanted. Call clear_tables(`all`) if you want different data in there
  warnings.warn("\nSome DJ tables (like UserInfo) are not empty, skipping adding new entries from config.\nMake sure this is wanted. Call clear_tables(`all`) if you want different data in there")


In [6]:
dj_table_holder.clear_tables("all")

[2025-10-26 20:49:09,984][INFO]: Deleting 3 rows from `ageuler_ssuhai_closed_loop`.`__field__stack_averages`
[2025-10-26 20:49:10,061][INFO]: Deleting 4 rows from `ageuler_ssuhai_closed_loop`.`__presentation__scan_info`
[2025-10-26 20:49:10,116][INFO]: Deleting 12 rows from `ageuler_ssuhai_closed_loop`.`__presentation__stack_averages`
[2025-10-26 20:49:10,284][INFO]: Deleting 90 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a__data_set`
[2025-10-26 20:49:10,366][INFO]: Deleting 90 rows from `ageuler_ssuhai_closed_loop`.`__fit_gauss2_d_r_f`
[2025-10-26 20:49:10,400][INFO]: Deleting 90 rows from `ageuler_ssuhai_closed_loop`.`__split_r_f`
[2025-10-26 20:49:10,507][INFO]: Deleting 90 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a`
[2025-10-26 20:49:10,585][INFO]: Deleting 90 rows from `ageuler_ssuhai_closed_loop`.`__d_noise_trace`
[2025-10-26 20:49:10,748][INFO]: Deleting 88 rows from `ageuler_ssuhai_closed_loop`.`__celltype_assignment`
[2025-10-26 20:49:10,784][INFO]: Deleting 88 rows fr

No ROI mask files found to delete.


In [8]:
rm_all_experiment_dirs(cfg.DJ.userinfo.data_dir)

Removed directory and all its contents: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/data_dj_format/20251008


In [12]:
dj_table_holder.setup()


[2025-10-26 20:51:50,667][INFO]: DataJoint is configured from /gpfs01/euler/User/ssuhai/datajoint/dj_ssuhai_conf.json


schema_name: ageuler_ssuhai_closed_loop


/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:195: UserWarning: Stimulus offset not set. Assuming 0 offset. This is incorrect for the standard dense noise stimulus.
  warnings.warn(
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:203: UserWarning: Stimulus offset not set. Assuming 0 offset. This is incorrect for the standard dense noise stimulus.
  warnings.warn(
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:112: UserWarning: Values for ['bardx', 'bardy', 'velumsec', 'tmovedurs'] in `stim_dict` for stimulus `movingbar` are None. This may cause problems downstream.
  warnings.warn(f'Values for {missing_info} in `stim_dict` for stimulus `{stim_name}` are None. '
/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/core/stimulus.py:39: UserWarning: Number of triggers in trial_info=8 must match ntrigger_rep=1.
  warnings.warn(msg)


preprocessing params:
 [{'preprocess_id': 1, 'fs_resample': 60, 'stim_names': ['gChirp', 'lChirp', 'movingbar', 'densenoise']}, {'preprocess_id': 2, 'window_length': 60, 'poly_order': 3, 'non_negative': 1, 'subtract_baseline': 0, 'standardize': 1, 'stim_names': ['mouse_cam']}]
Saving classifier to /gpfs01/euler/User/ssuhai/datajoint/rgc_classifier/rgc_classifier.pkl
Done setting up!


In [7]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    cfg=cfg,
    seeds=[111,222]
)

## During the experimet

### Move files from server to the repo 

In [14]:
create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,)

copy_rec_files(
    recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
    destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
    full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
)

COPIED file from /gpfs01/euler/data/Data/Suhai/test_data_klaudia/M1_LR_GCL0_Chirp_iter0.smh to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/data_dj_format/20251026/1/Raw/M1_LR_GCL0_Chirp_iter0.smh
COPIED file from /gpfs01/euler/data/Data/Suhai/test_data_klaudia/M1_LR_GCL0_MB_iter0.smh to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/data_dj_format/20251026/1/Raw/M1_LR_GCL0_MB_iter0.smh
COPIED file from /gpfs01/euler/data/Data/Suhai/test_data_klaudia/M1_LR_GCL0_MC18_iter0.smh to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/data_dj_format/20251026/1/Raw/M1_LR_GCL0_MC18_iter0.smh
COPIED file from /gpfs01/euler/data/Data/Suhai/test_data_klaudia/M1_LR_GCL0_Chirp_iter0.smp to /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/data_dj_format/20251026/1/Raw/M1_LR_GCL0_Chirp_iter0.smp
COPIED file from /gpfs01/euler/data/Data/Suhai/test_data_klaudia/M1_LR_GCL0_

### Preprocessing

In [15]:
preprocessor.upload_iteration_metadata()

Scanning for experimenter: closedlooptest
	header_path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/data/data_dj_format/20251026/1
		header_name: 20251026_left.ini
		Adding: {'experimenter': 'closedlooptest', 'date': datetime.datetime(2025, 10, 26, 0, 0), 'exp_num': 1}


OpticDisk: 100%|██████████| 1/1 [00:00<00:00, 28.33it/s]

Found 4 files in 1 fields for key={'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 26), 'exp_num': 1, 'raw_id': 1}


	Adding field: `{'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 26), 'exp_num': 1, 'raw_id': 1}`


Processes: 100%|██████████| 6/6 [00:08<00:00,  1.34s/it]


In [9]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
if len(missing_keys) == 1:
    field_key = missing_keys[0]
    print(f"Missing field key found: {field_key}")
elif len(missing_keys) > 1:
    raise ValueError(f"Multiple missing fields found: {missing_keys}")
else:
    print("No missing fields found, using the last field key.")
    all_field_key = dj_table_holder("Field")().proj().fetch(as_dict=True)
    field_key = all_field_key[-1]
    print(f"Field key: {field_key}")

No missing fields found, using the last field key.
Field key: {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 26), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0'}


In [17]:
# compute 
preprocessor.add_iteration_roi_mask(field_key=field_key)
preprocessor.add_iteration_rois()
preprocessor.add_iteration_traces()


field_key: {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 26), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0'} 
pres_keys: [{'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 26), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'stim_name': 'densenoise', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 26), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'stim_name': 'gChirp', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 26), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'stim_name': 'mouse_cam', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 10, 26), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'stim_name': 'movingbar', 'cond2': 'control'}]
No ROI masks found for field: {'experim

Returned InteractiveRoiCanvas object. To start GUI, call <enter_object_name>.start_gui().
Executing autorois for main stim idx 0 which is {'stim_name': 'gChirp', 'cond2': 'control'}


Skipping auto shift for main stim {'stim_name': 'gChirp', 'cond2': 'control'}
Auto shifting for stim {'stim_name': 'movingbar', 'cond2': 'control'}


Auto shifting for stim {'stim_name': 'mouse_cam', 'cond2': 'control'}


Auto shifting for stim {'stim_name': 'densenoise', 'cond2': 'control'}


Processes: 100%|██████████| 428/428 [00:00<00:00, 558.64it/s]


### qualty and RF

In [11]:
quality_type_analysis_wrapper.compute_analysis(
    field_key=field_key)

# filter 
passed_roi_ids_chirp_mb = quality_type_analysis_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
    d_qi_min =cfg.quality_filtering["d_qi_min"],
    qidx_min=cfg.quality_filtering["chirp_qi_min"],
    celltypes=cfg.quality_filtering["celltypes"],
    classifier_confidence= 0) # NOTE: this cfg.quality_filtering["classifier_confidence"])
if len(passed_roi_ids_chirp_mb) == 0:
    raise ValueError("No ROIs passed the quality criterion for quality and type.")
print(f"{len(passed_roi_ids_chirp_mb)} ROIs passed quality chirp mb filtering: {passed_roi_ids_chirp_mb}")



Found 78 rois passing the criterion out of 107 rois.                (d_qi_min=0.6, chrip qidx_min=0.35, celltypes=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32], classifier_confidence=0)
78 ROIs passed quality chirp mb filtering: [4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 21, 22, 24, 25, 26, 27, 28, 29, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 67, 68, 69, 71, 72, 73, 74, 75, 78, 80, 81, 82, 84, 86, 89, 91, 92, 93, 94, 96, 97, 99, 101, 102, 103, 106]


/gpfs01/euler/User/ssuhai/GitRepos/djimaging/djimaging/tables/classifier/rgc_classifier.py:402: UserWarning: Parallel processing not implemented!
  warnings.warn('Parallel processing not implemented!')


In [12]:
# sta 
sta_wrapper.compute_analysis(
    field_key=field_key,
    roi_id_subset=passed_roi_ids_chirp_mb,)


experimenter='closedlooptest' AND date='2025-10-26' AND exp_num='1' AND raw_id='1' AND field='GCL0' AND region='LR' AND cond1='iter0' AND roi_id in (4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 21, 22, 24, 25, 26, 27, 28, 29, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 67, 68, 69, 71, 72, 73, 74, 75, 78, 80, 81, 82, 84, 86, 89, 91, 92, 93, 94, 96, 97, 99, 101, 102, 103, 106)


Processes: 100%|██████████| 78/78 [00:01<00:00, 65.14it/s]


In [14]:
assert ((dj_table_holder("STA")() & field_key).fetch("roi_id") == passed_roi_ids_chirp_mb).all(), "STA roi_id does not match passed roi_ids from quality and type filtering."

In [15]:
# filter
passed_roi_ids_sta = sta_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
                                                               rf_qidx_min=0)# cfg.quality_filtering["rf_qidx_min"])
if len(passed_roi_ids_sta) == 0:
    raise ValueError("No ROIs passed the quality criterion for STA.")
print(f"{len(passed_roi_ids_sta)} ROIs passed STA filtering with rf_qidx_min={cfg.quality_filtering["rf_qidx_min"]}: {passed_roi_ids_sta}")

if len(passed_roi_ids_sta) < 25:
    raise ValueError(f"Less than 25 ROIs passed the quality criterion for STA. Found {len(passed_roi_ids_sta)} ROIs. Consider lowering the rf_qidx_min threshold.")


78 ROIs passed STA filtering with rf_qidx_min=0.5: [4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 21, 22, 24, 25, 26, 27, 28, 29, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 67, 68, 69, 71, 72, 73, 74, 75, 78, 80, 81, 82, 84, 86, 89, 91, 92, 93, 94, 96, 97, 99, 101, 102, 103, 106]


In [ ]:
if len(passed_roi_ids_sta) >= 25:
    print("MORE THAN 25 ROIS, selecting 25 best")
    top_n_rois_sta,top_n_scores = sta_wrapper.list_top_n_rois_by_qidx(field_key=field_key,
                                                    n=25,)
    passed_roi_ids_sta = top_n_rois_sta
    print(top_n_rois_sta)
    print(top_n_scores)
    print(f"Using top 25 rois based on rf_qidx: {passed_roi_ids_sta}")


# if len(passed_roi_ids_sta) >= 25:
#     print("MORE THAN 25 ROIS, selecting 25 best")
#     top_n_rois_sta,top_n_scores = sta_wrapper.list_top_n_rois_by_qidx(field_key=field_key,
#                                                     n=35,)
#     passed_roi_ids_sta = top_n_rois_sta
#     print(top_n_rois_sta)
#     print(top_n_scores)
#     print(f"Using top 25 rois based on rf_qidx: {passed_roi_ids_sta}")

### MEI

In [17]:
# compute
random_seed_mei_wrapper.load_all_data_from_dir(
    "/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis/GCL0_20251003_200456_full_dataset_single_member"
)

Loaded raw session dict from /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis/GCL0_20251003_200456_full_dataset_single_member/session_dict_raw.pkl
Loaded neuron data dict from /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis/GCL0_20251003_200456_full_dataset_single_member/neuron_data_dict.pkl
Loaded MEI data container from /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis/GCL0_20251003_200456_full_dataset_single_member/mei_data_container.pkl
Loaded full model from /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis/GCL0_20251003_200456_full_dataset_single_member/model_full.pt
Loaded metadata from /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis/GCL0_20251003_200456_full_dataset_single_member/metadata.pkl, with keys: ['roi2readout_idx_wmeis', 'roi_ids2readout_idx', 'neuron_idxs_passin

In [ ]:
### to adjust testset correl threshold
# random_seed_mei_wrapper.neuron_testset_correls
# # random_seed_mei_wrapper.quality_filtering["min_testset_correl"] = 0
# # random_seed_mei_wrapper.apply_quality_filter()
# # random_seed_mei_wrapper.mei_subanalysis()

## visualize with GUI

In [ ]:
from model_in_the_loop.core.gui import ExtendedAutoRoiGui
gui = ExtendedAutoRoiGui(
    dj_table_holder=dj_table_holder, 
    dj_preprocessor=preprocessor,
    all_dj_wrappers= [quality_type_analysis_wrapper,sta_wrapper,random_seed_mei_wrapper],
    # JUST VIS
    do_not_compute_only_visualize = True,
    
    field_key=field_key,
    canvas_width=30)

In [17]:
display(gui.start_gui())

### RF

In [ ]:
f,a = show_all_rois_plot(
    dj_table_holder=dj_table_holder,
    wrapper=sta_wrapper,
    field_key=field_key
    )

### MEI

In [ ]:
f,a = show_all_rois_plot(
    dj_table_holder=dj_table_holder,
    wrapper=random_seed_mei_wrapper,
    field_key=field_key
    )

# The stimulus

## select roi ids

In [ ]:
throw_out_these_rois = []

In [ ]:
only_these_rois = [45,77,82,] + [52,51,2]

In [ ]:
if only_these_rois != []:
    roi_ids_list = only_these_rois
    print(f"Using only these rois: {roi_ids_list}")
else:
    roi_ids_list = [roi for roi in passed_roi_ids_sta if roi not in throw_out_these_rois]
    print(f"Final roi ids list length: {len(roi_ids_list)}")

### RFs

In [ ]:
create_rf_test_dir(
    roi_ids_list= [40,45,62,77,82],
    stimulus_table=dj_table_holder("Stimulus")(),
    fit_gauss_2d_rf_table=dj_table_holder("FitGauss2DRF")(),
    abs_save_dir=cfg.paths.stimulus_output_dir,
    field_restriction= field_key,
)

In [ ]:
roi_id2mei_ids,roi_id2mei_info = random_seed_mei_wrapper.select_subset_of_meis_for_each_roi(
    only_consider_these_rois=roi_ids_list,
    neuron_data_dict = random_seed_mei_wrapper.neuron_data_dict,
    new_session_id = random_seed_mei_wrapper.new_session_id,
    mei_data_container = random_seed_mei_wrapper.mei_data_container,
    readout_idx_wmei2rois = random_seed_mei_wrapper.readout_idx_wmei2rois,
    n_stimuli_total = 6,
    )
print(roi_id2mei_ids)

### meis

In [ ]:
create_single_mei_avis_and_metadata(
    rois_selected=roi_ids_list,
    roi_id2mei_ids = roi_id2mei_ids,    
    mei_data_container=random_seed_mei_wrapper.mei_data_container,
    stimulus_table=dj_table_holder("Stimulus")(),
    fit_gauss_2d_rf_table= dj_table_holder("FitGauss2DRF")(),
    abs_save_dir=cfg.paths.stimulus_output_dir,
    field_restriction=field_key,
)



# Save data

### RFs

In [ ]:
import datetime
now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_rf_dir = os.path.join(cfg.paths.repo_directory,"model_in_the_loop/data/online_computed_data",f"rf_test_stimuli_{now}")
os.makedirs(save_rf_dir,exist_ok=True)

create_rf_test_dir(
    roi_ids_list= roi_ids_list,
    stimulus_table=dj_table_holder("Stimulus")(),
    fit_gauss_2d_rf_table=dj_table_holder("FitGauss2DRF")(),
    abs_save_dir=save_rf_dir
)

### meis

In [ ]:

random_seed_mei_wrapper.save_all_data_to_dir(save_dir_parent=random_seed_mei_wrapper.save_dir_parent)
# save data 
import datetime
now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_mei_dir = os.path.join(cfg.paths.repo_directory,"model_in_the_loop/data/online_computed_data",f"mei_test_stimuli_{now}")
os.makedirs(save_mei_dir,exist_ok=True)



# Clean up

In [ ]:
userinput = input("Cleanup? (y/n): ")
if userinput.lower() == 'y':
    dj_table_holder.clear_tables("all")


# shit goes south